# Qwen2.5-Math-1.5B + LoRA — Antrenament BAC

Fine-tune Qwen2.5-Math-1.5B-Instruct cu LoRA pe exercitii BAC romanesc.

## Setup Kaggle
1. **Add Data** → cauta `bac-prep-data` → adauga
2. **Add Model** → cauta `qwen2.5-math-1.5b-instruct` (alekseizabirnik) → adauga
3. **GPU**: Settings → Accelerator → GPU T4 x2
4. **Internet**: Settings → Internet → **ON**
5. **Run All**

In [ ]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 \
    accelerate>=0.30.0 bitsandbytes>=0.46.1 datasets>=2.19.0 \
    sentencepiece huggingface_hub

In [ ]:
import os, json, torch

# Gaseste datele automat
def find_file(root, filename):
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None

TRAIN_JSONL = find_file('/kaggle/input', 'train.jsonl')
VAL_JSONL = find_file('/kaggle/input', 'val.jsonl')
TEST_JSONL = find_file('/kaggle/input', 'test.jsonl')

print(f'Train: {TRAIN_JSONL}')
print(f'Val:   {VAL_JSONL}')
print(f'Test:  {TEST_JSONL}')

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl(TRAIN_JSONL)
val_data = load_jsonl(VAL_JSONL)
test_data = load_jsonl(TEST_JSONL)

print(f'\nTrain: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print(f'\nSample:\n{train_data[0]["text"][:300]}...')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_PATH = '/kaggle/input/models/alekseizabirnik/qwen2.5-math-1.5b-instruct/transformers/default/1'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Incarc Qwen2.5-Math-1.5B (4-bit)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True, local_files_only=True,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH, trust_remote_code=True, padding_side='right', local_files_only=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f'Parametri: {model.num_parameters():,}')
print(f'VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset as HFDataset

MAX_SEQ_LEN = 512

def tokenize_sample(sample):
    result = tokenizer(sample['text'], truncation=True, max_length=MAX_SEQ_LEN, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result

train_ds = HFDataset.from_list(train_data).map(tokenize_sample, remove_columns=['text'])
val_ds = HFDataset.from_list(val_data).map(tokenize_sample, remove_columns=['text'])
train_ds.set_format('torch')
val_ds.set_format('torch')

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = '/kaggle/working/qwen-bac-lora'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    weight_decay=0.01,
    fp16=False, bf16=False,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=10,
    logging_first_step=True,
    report_to='none',
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    max_grad_norm=1.0,
    seed=42,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
)

print(f'Effective batch size: {4 * 4}')
print(f'Total steps: ~{len(train_ds) // 16 * 3}')
print('Incep antrenamentul...')

train_result = trainer.train()

print(f'\n{"="*60}')
print(f'Train loss: {train_result.metrics["train_loss"]:.4f}')
print(f'Runtime: {train_result.metrics["train_runtime"]:.0f}s')
print(f'{"="*60}')

In [ ]:
# Evaluare test
test_ds = HFDataset.from_list(test_data).map(tokenize_sample, remove_columns=['text'])
test_results = trainer.evaluate(eval_dataset=test_ds)
print(f'Test loss: {test_results["eval_loss"]:.4f}')
print(f'Test perplexity: {torch.exp(torch.tensor(test_results["eval_loss"])):.2f}')

# Salvare adapter
ADAPTER_PATH = '/kaggle/working/qwen-bac-lora/best_adapter'
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'\nAdapter salvat: {ADAPTER_PATH}')

In [ ]:
# Test inferenta
SYSTEM_PROMPT = (
    'Ești un asistent de matematică specializat pe exerciții BAC. '
    'Rezolvă exercițiul pas cu pas, explicând fiecare etapă.'
)

test_questions = [
    'Rezolvați ecuația: 2x + 3 = 7',
    'Calculați derivata funcției f(x) = x^3 - 6x^2 + 9x + 1',
    'Calculați limita: lim(x->inf) (x^2 + 3x) / (2x^2 - 1)',
    'Calculați determinantul matricei A = [[3, 1], [2, 4]]',
]

model.eval()
for q in test_questions:
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n'
        f'<|im_start|>user\n{q}\n<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256, temperature=0.1,
            top_k=30, do_sample=True, pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    if '<|im_end|>' in response:
        response = response[:response.index('<|im_end|>')]
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'A: {response.strip()}')

In [ ]:
# Curbe antrenament
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_steps = [h['step'] for h in log_history if 'loss' in h]
train_losses = [h['loss'] for h in log_history if 'loss' in h]
eval_steps = [h['step'] for h in log_history if 'eval_loss' in h]
eval_losses = [h['eval_loss'] for h in log_history if 'eval_loss' in h]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label='Train Loss', alpha=0.7)
plt.plot(eval_steps, eval_losses, 'ro-', label='Val Loss', markersize=4)
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Qwen2.5-Math LoRA Fine-tuning')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig('/kaggle/working/lora_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Push to HuggingFace
from huggingface_hub import login, HfApi

# ==============================
# CONFIGUREAZA AICI
# ==============================
HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXX'  # Settings -> Access Tokens -> New token (Write)
HF_REPO = 'USERNAME/bac-math-qwen-lora'  # Pune username-ul tau

login(token=HF_TOKEN)
api = HfApi()

api.create_repo(repo_id=HF_REPO, repo_type='model', private=True, exist_ok=True)

# Fix: schimba base_model din path local Kaggle in ID-ul HuggingFace
import json as _json
adapter_config_path = os.path.join(ADAPTER_PATH, 'adapter_config.json')
if os.path.exists(adapter_config_path):
    with open(adapter_config_path, 'r') as f:
        cfg = _json.load(f)
    cfg['base_model_name_or_path'] = 'Qwen/Qwen2.5-Math-1.5B-Instruct'
    with open(adapter_config_path, 'w') as f:
        _json.dump(cfg, f, indent=2)
    print('Fixed adapter_config.json base_model path')

# Fix: sterge README.md generat automat (poate avea YAML invalid)
readme_path = os.path.join(ADAPTER_PATH, 'README.md')
if os.path.exists(readme_path):
    os.remove(readme_path)
    print('Removed auto-generated README.md')

print('Upload adapter LoRA...')
api.upload_folder(
    folder_path=ADAPTER_PATH, repo_id=HF_REPO,
    path_in_repo='best_adapter',
    commit_message='Upload LoRA adapter BAC',
)

if os.path.exists('/kaggle/working/lora_training_curves.png'):
    api.upload_file(
        path_or_fileobj='/kaggle/working/lora_training_curves.png',
        path_in_repo='lora_training_curves.png', repo_id=HF_REPO,
    )

print(f'\nDone! https://huggingface.co/{HF_REPO}')